<a href="https://colab.research.google.com/github/wyldescience/FolSum/blob/main/Model%20training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

YOLO model to count and detect *Folsomia candida* nymphs in arenas (charcoal + plaster and paris background

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install ultralytics opencv-python


***Arena crop (remove outside background to leave circular container) + tiling exporter (YOLO-friendly)***




*   Reads full images
*   Crops arena from background


*   optionally resizes

*   tiles into tile_size with overlap
*   writes tiles to tiles/images/


*   writes a tiles_manifest.csv so can map tiles back to source image





In [ ]:
import os, glob, csv
import cv2
import numpy as np

ROOT = "ROOT DIRECTORY"
IN_FULL = os.path.join(ROOT, "images_full")
OUT_ARENA = os.path.join(ROOT, "arena_cropped")
OUT_TILES_IMG = os.path.join(ROOT, "tiles", "images")
OUT_TILES_LBL = os.path.join(ROOT, "tiles", "labels")

os.makedirs(OUT_ARENA, exist_ok=True)
os.makedirs(OUT_TILES_IMG, exist_ok=True)
os.makedirs(OUT_TILES_LBL, exist_ok=True)

# -------- arena crop (your circle-fit) --------
def crop_arena_by_contour(img_rgb, pad=15):
    img_rgb = img_rgb.astype(np.uint8)
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    blur = cv2.GaussianBlur(gray, (9, 9), 0)
    _, mask = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    k1 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (31, 31))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k1)
    k2 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k2)

    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return img_rgb

    H, W = gray.shape
    cx0, cy0 = W/2, H/2

    best, best_score = None, -1e18
    for c in cnts:
        area = cv2.contourArea(c)
        if area < 0.05 * H * W:
            continue
        peri = cv2.arcLength(c, True)
        if peri <= 0:
            continue
        circularity = 4 * np.pi * area / (peri * peri)
        (x, y), r = cv2.minEnclosingCircle(c)
        dist = ((x - cx0)**2 + (y - cy0)**2) ** 0.5
        score = (circularity * 2.0) + (area / (H*W)) - (dist / max(H, W))
        if score > best_score:
            best_score, best = score, c

    if best is None:
        best = max(cnts, key=cv2.contourArea)

    (x, y), r = cv2.minEnclosingCircle(best)
    x, y, r = int(x), int(y), int(r)

    arena_mask = np.zeros((H, W), dtype=np.uint8)
    cv2.circle(arena_mask, (x, y), r, 255, thickness=-1)

    x0 = max(0, x - r - pad); x1 = min(W, x + r + pad)
    y0 = max(0, y - r - pad); y1 = min(H, y + r + pad)

    cropped = img_rgb[y0:y1, x0:x1].copy()
    m = arena_mask[y0:y1, x0:x1] > 0
    cropped[~m] = 0
    return cropped

def tile_image(img_rgb, tile_size=640, overlap=0.2):
    H, W = img_rgb.shape[:2]
    stride = int(tile_size * (1 - overlap))
    stride = max(1, stride)

    tiles = []
    for y in range(0, H, stride):
        for x in range(0, W, stride):
            y2 = min(y + tile_size, H)
            x2 = min(x + tile_size, W)
            y1 = max(0, y2 - tile_size)
            x1 = max(0, x2 - tile_size)
            tile = img_rgb[y1:y2, x1:x2]
            tiles.append((tile, x1, y1, x2, y2))
    return tiles

# -------- export --------
tile_size = 640
overlap = 0.25
resize_long_side = 2400  # speeds training; preserves nymph detail usually. set None to skip.

manifest_path = os.path.join(ROOT, "tiles_manifest.csv")
with open(manifest_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["tile_file","src_file","x1","y1","x2","y2","tile_w","tile_h"])

    img_paths = sorted(glob.glob(os.path.join(IN_FULL, "*.*")))
    img_paths = [p for p in img_paths if p.lower().endswith((".jpg",".jpeg",".png",".tif",".tiff"))]

    for p in img_paths:
        bgr = cv2.imread(p)
        if bgr is None:
            print("skip unreadable:", p); continue
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

        arena = crop_arena_by_contour(rgb, pad=15)

        # optional resize
        if resize_long_side is not None:
            H, W = arena.shape[:2]
            scale = resize_long_side / max(H, W)
            if scale < 1.0:
                arena = cv2.resize(arena, (int(W*scale), int(H*scale)), interpolation=cv2.INTER_AREA)

        # save arena crop for reference
        base = os.path.splitext(os.path.basename(p))[0]
        arena_path = os.path.join(OUT_ARENA, f"{base}_arena.jpg")
        cv2.imwrite(arena_path, cv2.cvtColor(arena, cv2.COLOR_RGB2BGR), [int(cv2.IMWRITE_JPEG_QUALITY), 95])

        tiles = tile_image(arena, tile_size=tile_size, overlap=overlap)

        for idx, (tile, x1, y1, x2, y2) in enumerate(tiles):
            # skip tiles that are mostly black (outside arena)
            if (tile.sum() / (tile.size*255)) < 0.02:
                continue

            tile_file = f"{base}__t{idx:04d}__x{x1}_y{y1}.jpg"
            out_tile_path = os.path.join(OUT_TILES_IMG, tile_file)
            cv2.imwrite(out_tile_path, cv2.cvtColor(tile, cv2.COLOR_RGB2BGR), [int(cv2.IMWRITE_JPEG_QUALITY), 95])

            writer.writerow([tile_file, os.path.basename(p), x1, y1, x2, y2, tile.shape[1], tile.shape[0]])

print("Done. Tiles in:", OUT_TILES_IMG)
print("Manifest:", manifest_path)


# Create YOLO train/validation split + data.yaml

assumes labels are YOLO text files in tiles/labels with same basename as tile image


In [ ]:
import os, random, shutil, glob

ROOT = "ROOT DIRECTORY"
IMG_DIR = os.path.join(ROOT, "tiles", "images")
LBL_DIR = os.path.join(ROOT, "tiles", "labels")

# YOLO expects:
# datasets/myset/images/train, images/val, labels/train, labels/val
DS = os.path.join(ROOT, "dataset")
for sub in ["images/train","images/val","labels/train","labels/val"]:
    os.makedirs(os.path.join(DS, sub), exist_ok=True)

imgs = sorted(glob.glob(os.path.join(IMG_DIR, "*.jpg")) + glob.glob(os.path.join(IMG_DIR, "*.png")))

pairs = []
for im in imgs:
    stem = os.path.splitext(os.path.basename(im))[0]
    lab = os.path.join(LBL_DIR, stem + ".txt")
    pairs.append((im, lab))  # label may or may not exist

print("Total tiles:", len(pairs))
print("With labels:", sum(os.path.exists(l) for _, l in pairs))
print("No labels:",  sum(not os.path.exists(l) for _, l in pairs))


random.seed(42)
random.shuffle(pairs)
split = int(0.8 * len(pairs))
train = pairs[:split]
val = pairs[split:]


def copy_pair(pairs, splitname):
    for im, lab in pairs:
        shutil.copy2(im, os.path.join(DS, "images", splitname, os.path.basename(im)))

        out_lab = os.path.join(DS, "labels", splitname, os.path.splitext(os.path.basename(im))[0] + ".txt")
        if os.path.exists(lab):
            shutil.copy2(lab, out_lab)
        else:
            open(out_lab, "w").close()   # empty label file = background tile

copy_pair(train, "train")
copy_pair(val, "val")

yaml_path = os.path.join(ROOT, "springtails.yaml")
with open(yaml_path, "w") as f:
    f.write(f"path: {DS}\n")
    f.write("train: images/train\n")
    f.write("val: images/val\n")
    f.write("names:\n")
    f.write("  0: nymph\n")

print("Wrote:", yaml_path)
print("Train tiles:", len(train), "Val tiles:", len(val))


Train YOLOv8 (tiny model, fast)


In [ ]:
from ultralytics import YOLO
import torch

print("GPU:", torch.cuda.is_available())

yaml_path = os.path.join(ROOT, "springtails.yaml")
runs_dir = os.path.join(ROOT, "yolo_runs")

model = YOLO("yolov8n.pt")  # nano = fastest

model.train(
    data=yaml_path,
    imgsz=640,
    epochs=40,
    batch=16,
    workers=2,
    project=runs_dir,
    name="nymph_yolov8n_tiles",
    patience=10
)


Check results from nano model

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

img = Image.open(
    "/content/drive/MyDrive/YOLO nymph detector/yolo_runs/nymph_yolov8n_tiles/results.png"
)
plt.figure(figsize=(12,8))
plt.imshow(img)
plt.axis("off")


import pandas as pd

df = pd.read_csv(
    "/content/drive/MyDrive/YOLO nymph detector/yolo_runs/nymph_yolov8n_tiles/results.csv"
)
df.tail(10)


val_imgs = sorted(glob.glob(
    "/content/drive/MyDrive/YOLO nymph detector/yolo_runs/nymph_yolov8n_tiles/val_batch*.jpg"
))

plt.figure(figsize=(12,6))
plt.imshow(Image.open(val_imgs[0]))
plt.axis("off")

# Inference on a full arena: Tile --> detect --> merge --> count



*   Crop arena


*   tile it

*   run detections on each tile
*   shift boxes back into arena coordinates


*   perform simple NMS to merge duplicates from overlap

*   Output an overlay & count







In [ ]:
import os, glob
import cv2
import numpy as np
from ultralytics import YOLO

def nms_xyxy(boxes, scores, iou_thr=0.4):
    if len(boxes) == 0:
        return []
    boxes = np.array(boxes, dtype=np.float32)
    scores = np.array(scores, dtype=np.float32)
    x1,y1,x2,y2 = boxes[:,0],boxes[:,1],boxes[:,2],boxes[:,3]
    areas = (x2-x1+1)*(y2-y1+1)
    order = scores.argsort()[::-1]
    keep=[]
    while order.size>0:
        i = order[0]
        keep.append(i)
        xx1 = np.maximum(x1[i], x1[order[1:]])
        yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]])
        yy2 = np.minimum(y2[i], y2[order[1:]])
        w = np.maximum(0, xx2-xx1+1)
        h = np.maximum(0, yy2-yy1+1)
        inter = w*h
        iou = inter / (areas[i] + areas[order[1:]] - inter + 1e-6)
        inds = np.where(iou <= iou_thr)[0]
        order = order[inds+1]
    return keep

# reuse crop_arena_by_contour + tile_image from earlier cell
# (make sure those functions exist in your notebook)

runs_dir = os.path.join(ROOT, "yolo_runs", "nymph_yolov8n_tiles")
weights = os.path.join(runs_dir, "weights", "best.pt")
model = YOLO(weights)

def infer_count_on_full_image(full_img_path, tile_size=640, overlap=0.25, conf=0.25, iou_merge=0.4):
    bgr = cv2.imread(full_img_path)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    arena = crop_arena_by_contour(rgb, pad=15)

    tiles = tile_image(arena, tile_size=tile_size, overlap=overlap)

    all_boxes=[]
    all_scores=[]

    for (tile, x1, y1, x2, y2) in tiles:
        if (tile.sum() / (tile.size*255)) < 0.02:
            continue
        res = model.predict(tile, imgsz=tile_size, conf=conf, verbose=False)[0]
        if res.boxes is None or len(res.boxes) == 0:
            continue
        b = res.boxes.xyxy.cpu().numpy()
        s = res.boxes.conf.cpu().numpy()
        # shift to arena coords
        b[:,[0,2]] += x1
        b[:,[1,3]] += y1
        all_boxes.extend(b.tolist())
        all_scores.extend(s.tolist())

    keep = nms_xyxy(all_boxes, all_scores, iou_thr=iou_merge)
    kept_boxes = [all_boxes[i] for i in keep]

    ann = arena.copy()
    for (x1,y1,x2,y2) in kept_boxes:
        cv2.rectangle(ann, (int(x1),int(y1)), (int(x2),int(y2)), (255,0,0), 2)

    return arena, ann, len(kept_boxes)

# test on one image
test_img = "TEST IMG JPG"
arena, ann, count = infer_count_on_full_image(test_img, conf=0.25)
print("Count:", count)

import matplotlib.pyplot as plt
plt.figure(figsize=(14,6))
plt.subplot(1,2,1); plt.imshow(arena); plt.title("Arena"); plt.axis("off")
plt.subplot(1,2,2); plt.imshow(ann); plt.title(f"Detections (count={count})"); plt.axis("off")
plt.show()
